# ✅ 분류(Classification) 문제 템플릿 v2

## 🍎 이 파일, 언제 쓰나요?
```
정답(Target)이 "종류/범주"일 때 사용합니다.

  "생존 여부를 예측"     → 생존/사망 (2종류)         → ✅ 분류
  "신용등급을 예측"      → Good/Standard/Poor (3종류) → ✅ 분류 (다중클래스!)
  "이동 시간을 예측"     → 423초 같은 숫자            → ❌ 회귀 템플릿 사용

판정 기준: 평가지표가 accuracy / f1_score / roc_auc 면 분류입니다.
```

## v2에서 달라진 점 (기존 템플릿 문제점 보완)
| 문제 | v2 개선 |
|---|---|
| 어떤 컬럼을 어떻게 처리할지 모르겠음 | `analyze_columns()`가 삭제/OHE/라벨인코딩 리스트를 **직접 만들어 반환** |
| 정답이 3종류 이상(다중클래스)이면 막막함 | Step2에 클래스 개수 자동 확인 + `f1_score(average='macro')`로 통일 |
| 설명이 어려움 | 각 단계마다 "왜 이렇게 하는지" 이유를 함께 설명 |
| 성능을 더 올리고 싶음 | Step7에 교차검증·불균형 보정·앙상블 등 추가 |

## 전체 흐름
```
[Step1] 도구 꺼내기
[Step2] 데이터 보기 (EDA) + 정답 종류(이진/다중) 확인
[Step3] 데이터 손질 (전처리: 삭제 / 결측치 / 인코딩 / 스케일링)
[Step4] AI 모델 학습 + 평가 (f1-macro 기준)
[Step5] 테스트 데이터 예측 + 혼동행렬로 확인
[Step6] 제출 파일 만들기
[Step7] 성능 더 올리기 (선택)
```

## ★ 표시된 곳만 실제 문제에 맞게 바꾸면 됩니다!


---
## [Step1] 도구 꺼내기

```
요리 전에 냄비·칼·도마를 꺼내듯, 코딩 전에 라이브러리를 먼저 불러옵니다.
lightgbm이 설치되어 있지 않다면 아래 셀의 주석을 풀고 설치하세요.
```

In [ ]:
# 필요하면 주석 해제 후 실행 (이미 설치되어 있으면 생략)
# pip install lightgbm xgboost


In [ ]:
# 기본 도구 (데이터 다루기)
import numpy as np
import pandas as pd

# 그래프 그리기
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')  # 경고 메시지 끄기


In [ ]:
# ★ 분류 모델들 (이 중에서 가장 좋은 것을 골라 씁니다)
from sklearn.linear_model import LogisticRegression      # 로지스틱 회귀
from sklearn.tree import DecisionTreeClassifier          # 결정 트리
from sklearn.neighbors import KNeighborsClassifier       # KNN
from sklearn.ensemble import RandomForestClassifier      # 랜덤 포레스트
from sklearn.ensemble import ExtraTreesClassifier        # 엑스트라 트리
from sklearn.ensemble import VotingClassifier            # 앙상블(투표)
from xgboost import XGBClassifier                        # XGBoost
from lightgbm import LGBMClassifier                      # LightGBM

# 데이터 나누기 / 인코딩 / 스케일링
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# ★ 평가지표 — 다중클래스까지 안전하게 쓰려면 f1_score(average='macro')를 기본으로 사용
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay, classification_report
)


### 데이터 불러오기

In [ ]:
train = pd.read_csv('./data/train.csv')  # 정답이 있는 학습용 데이터
test  = pd.read_csv('./data/test.csv')   # 정답 없는 예측용 데이터

print('train 크기:', train.shape)
print('test  크기:', test.shape)
train.head()


In [ ]:
train.info()


In [ ]:
test.info()


---
## [Step2] 데이터 보기 (EDA)

### (0) ★ 정답(Target) 컬럼 지정 + 몇 종류인지 확인 — 다중클래스 대응

```
분류 문제는 정답이 딱 2종류(이진분류)일 수도, 3종류 이상(다중분류)일 수도 있습니다.
아래 셀을 실행하면 자동으로 알려주고, 그에 맞는 평가지표 사용법을 알려줍니다.

⚠️ 다중분류인데 accuracy만 보면 안 되는 이유:
   클래스 비율이 90:5:5 처럼 불균형할 때, 다 90번 클래스로만 찍어도
   accuracy는 90%가 나옵니다. → 클래스별 성능을 고르게 반영하는
   f1_score(average='macro')를 기본 지표로 씁니다.
```

In [ ]:
TARGET = 'Credit_Score'  # ★ 정답 컬럼명 변경

n_classes = train[TARGET].nunique()
print(f'🎯 TARGET: {TARGET}')
print(f'클래스 개수: {n_classes}개 →', '이진분류(Binary)' if n_classes == 2 else '다중분류(Multi-class)')
print(train[TARGET].value_counts())

if n_classes == 2:
    print('\n💡 이진분류: accuracy, f1_score 아무거나 써도 괜찮지만 이 템플릿은 f1-macro로 통일합니다.')
else:
    print('\n💡 다중분류: f1_score(average=\'macro\')를 기본 평가지표로 사용합니다.')
    print('   XGBoost/LightGBM은 클래스 개수를 자동으로 감지해서 다중분류 모드로 동작하므로')
    print('   모델 코드를 따로 바꿀 필요는 없습니다.')


### (1) 컬럼 처리 방향 자동 판단 — ❌삭제 / 🔤OHE / 🏷라벨인코딩 / 📊스케일링

```
이 함수 하나가 "이 컬럼을 어떻게 요리할지" 판단부터 리스트 생성까지 다 해줍니다.

❌ 삭제        → test에 없는 컬럼(누수) / ID류(고유값=행수) / 결측률 너무 높음
🔤 OHE        → 범주형인데 종류가 적음(≤10) → 종류별로 0/1 컬럼을 새로 만듦
🏷 라벨인코딩  → 범주형인데 종류가 많음(11~50) → 숫자 하나로 치환(트리 모델은 순서 상관없이
                분기 기준으로만 사용하므로 무리 없이 사용 가능)
📊 스케일링    → 연속형 숫자 → 0~1 범위로 맞춤
📅 날짜        → 날짜형 컬럼은 자동으로 연/월/일/시로 쪼갠 뒤 다시 판단

★ 왜 OHE 대신 라벨인코딩을 쓰기도 하나요?
   범주가 50가지인 컬럼을 OHE 하면 컬럼이 50개 늘어나 학습이 느려지고
  차원의 저주(과적합 위험)가 생깁니다. 트리 계열 모델(RF/XGB/LGBM)은
  숫자로 바꾸기만 해도 알아서 기준값으로 잘 분기하기 때문에 라벨인코딩으로 충분합니다.
```

In [ ]:
def analyze_columns(train_df, test_df, target_col,
                     ohe_max=10, label_max=50, null_ratio_drop=0.5):
    """
    train_df, test_df : 학습/테스트 데이터프레임 (test_df는 누수 판단에만 사용)
    target_col        : 정답 컬럼명
    ohe_max           : 이 값 이하 고유값이면 OHE 추천
    label_max         : 이 값 이하 고유값이면 라벨인코딩 추천 (넘으면 삭제 검토)
    null_ratio_drop    : 결측 비율이 이보다 크면 삭제 추천

    반환값: drop_cols, ohe_cols, label_cols, num_cols (바로 코드에 쓸 수 있는 리스트)
    """
    drop_cols, ohe_cols, label_cols, num_cols = [], [], [], []

    print(f"{'컬럼명':28} {'타입':10} {'고유값':>7} {'결측수':>7}  처리 방향")
    print('-' * 90)

    for col in train_df.columns:
        if col == target_col:
            print(f"{col:28} {'':10} {'':>7} {'':>7}  🎯 TARGET (건드리지 않음)")
            continue

        dtype = str(train_df[col].dtype)
        n_unique = train_df[col].nunique()
        n_null = train_df[col].isnull().sum()
        null_ratio = n_null / len(train_df)

        # 1) test에 없는 컬럼 = 누수 위험 → 무조건 삭제
        if test_df is not None and col not in test_df.columns:
            drop_cols.append(col)
            direction = '❌ 삭제 (test에 없는 컬럼 → 누수 위험)'
        # 2) ID류: 고유값 수가 행 수와 거의 같음
        elif n_unique >= len(train_df) * 0.98 and dtype != 'float64':
            drop_cols.append(col)
            direction = '❌ 삭제 (ID류, 고유값이 행 수만큼 많음)'
        # 3) 결측 비율이 너무 높음
        elif null_ratio > null_ratio_drop:
            drop_cols.append(col)
            direction = f'❌ 삭제 (결측률 {null_ratio:.0%} — 채워도 신뢰하기 어려움)'
        # 4) 날짜형
        elif 'datetime' in dtype:
            direction = '📅 날짜형 → 연/월/일/시로 분리 후 다시 판단 필요'
        # 5) 연속 숫자형: pandas 버전에 따라 문자열이 'object' 대신 'str'로 표시되기도
        #    하므로 dtype 문자열 비교 대신 is_numeric_dtype()로 판단 (버전 호환)
        elif pd.api.types.is_numeric_dtype(train_df[col]) and n_unique > ohe_max:
            num_cols.append(col)
            direction = '📊 스케일링 대상 (연속 숫자)'
        # 6) 범주형 (문자열이거나, 숫자여도 종류가 적어서 카테고리로 보는 경우)
        else:
            if n_unique <= ohe_max:
                ohe_cols.append(col)
                direction = f'🔤 OHE 추천 (고유값 {n_unique}개 ≤ {ohe_max})'
            elif n_unique <= label_max:
                label_cols.append(col)
                direction = f'🏷 라벨인코딩 추천 (고유값 {n_unique}개, OHE는 너무 많음)'
            else:
                drop_cols.append(col)
                direction = f'❌ 삭제 검토 (고유값 {n_unique}개 — 텍스트/식별자 가능성)'

        null_mark = f'  ⚠️ 결측 {n_null}개' if n_null > 0 else ''
        print(f"{col:28} {dtype:10} {n_unique:>7} {n_null:>7}  {direction}{null_mark}")

    print()
    print(f'✔️ 삭제 {len(drop_cols)}개 | OHE {len(ohe_cols)}개 | 라벨인코딩 {len(label_cols)}개 | 스케일링 {len(num_cols)}개')
    return drop_cols, ohe_cols, label_cols, num_cols


drop_cols, ohe_cols, label_cols, num_cols = analyze_columns(train, test, target_col=TARGET)
print('\ndrop_cols =', drop_cols)
print('ohe_cols  =', ohe_cols)
print('label_cols=', label_cols)
print('num_cols  =', num_cols)


### (2) 위 리스트가 마음에 안 들면 여기서 직접 조정하세요

```
analyze_columns()는 "추천"일 뿐입니다. 도메인 지식으로 볼 때 다르게 판단되면
아래에서 리스트에 직접 추가/제거하세요. (예: 결측이 많아도 중요해 보이면 삭제 대신 채우기로 변경)
```

In [ ]:
# ★ 필요하면 여기서 직접 수정
# 예) drop_cols.remove('Occupation')       # 삭제 대상에서 빼기
# 예) label_cols.append('Occupation')      # 라벨인코딩 대상에 추가

print('최종 drop_cols  :', drop_cols)
print('최종 ohe_cols   :', ohe_cols)
print('최종 label_cols :', label_cols)
print('최종 num_cols   :', num_cols)


### (3) 숫자형 변수 분포 확인 (히스토그램)

In [ ]:
show_cols = num_cols[:8]  # 너무 많으면 앞 8개만
fig, axes = plt.subplots(1, len(show_cols), figsize=(4*len(show_cols), 3))
if len(show_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, show_cols):
    sns.histplot(train[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()


### (4) 범주형 변수 + 정답(Target) 분포 확인 (막대 그래프)

In [ ]:
cat_show = (ohe_cols + label_cols)[:6] + [TARGET]
fig, axes = plt.subplots(1, len(cat_show), figsize=(4*len(cat_show), 3))
if len(cat_show) == 1:
    axes = [axes]
for ax, col in zip(axes, cat_show):
    sns.countplot(x=train[col].astype(str), ax=ax)
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


---
## [Step3] 데이터 손질 (전처리)

### (1) 중복값 제거 — train만!

In [ ]:
print('중복 행 수:', train.duplicated().sum())
train = train.drop_duplicates().reset_index(drop=True)


### (2) 불필요한 컬럼 삭제 — analyze_columns의 drop_cols 적용

In [ ]:
train = train.drop(columns=drop_cols, errors='ignore')
test  = test.drop(columns=[c for c in drop_cols if c in test.columns], errors='ignore')
print('삭제 후 크기:', train.shape, test.shape)


### (3) 결측치(빈칸) 채우기

```
📊 숫자형(num_cols)   → train의 중앙값(median)으로 채우기 (평균보다 이상치에 덜 민감)
🔤🏷 범주형(ohe/label) → train의 최빈값(mode)으로 채우기

⚠️ 항상 train 기준으로 계산한 값을 test에도 그대로 적용합니다!
```

In [ ]:
print('=== train 결측치 ===')
print(train.isnull().sum()[train.isnull().sum() > 0])
print('=== test  결측치 ===')
print(test.isnull().sum()[test.isnull().sum() > 0])


In [ ]:
# 📊 숫자형 → train 중앙값으로 채우기
for c in num_cols:
    fill_value = train[c].median()
    train[c] = train[c].fillna(fill_value)
    if c in test.columns:
        test[c] = test[c].fillna(fill_value)

# 🔤🏷 범주형 → train 최빈값으로 채우기
for c in ohe_cols + label_cols:
    mode_series = train[c].mode(dropna=True)
    fill_value = mode_series[0] if len(mode_series) > 0 else 'Unknown'
    train[c] = train[c].fillna(fill_value)
    if c in test.columns:
        test[c] = test[c].fillna(fill_value)

print('train 결측치 총합:', train.isnull().sum().sum())
print('test  결측치 총합:', test.isnull().sum().sum())


### (4) 아웃라이어(이상치) 처리 — feature(입력값)만! (분류는 Target을 건드리지 않음)

```
분류의 정답은 범주(0/1, A/B/C)라서 이상치 개념이 없습니다.
입력값(num_cols) 중에 눈에 띄는 이상치가 있으면만 처리하세요. 없으면 이 셀은 건너뛰어도 됩니다.
```

In [ ]:
# 예시: IQR 방법으로 특정 숫자형 컬럼의 극단값을 경계값으로 눌러주기 (clip)
# 지우지 않고 clip하는 이유: 행을 통째로 지우면 test 예측 대상과 분포가 달라질 수 있음
for c in num_cols:
    Q1, Q3 = train[c].quantile(0.25), train[c].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    train[c] = train[c].clip(lower, upper)
    if c in test.columns:
        test[c] = test[c].clip(lower, upper)
print('이상치 clip 완료 (필요 없으면 이 셀은 무시해도 됩니다)')


### (5) Feature(입력) / Target(정답) 분리

In [ ]:
Xtrain = train.drop(columns=[TARGET])
ytrain = train[TARGET]
Xtest  = test.copy()

print('Xtrain:', Xtrain.shape, ' ytrain:', ytrain.shape, ' Xtest:', Xtest.shape)


### (6) 인코딩 — 🔤 OHE + 🏷 라벨인코딩

```
⚠️ train/test를 따로따로 get_dummies 하면 서로 다른 컬럼이 생길 수 있습니다
   (예: train에는 'C등급'이 있는데 test에는 없으면 컬럼 개수가 달라짐).
   그래서 train+test를 합쳐서 한 번에 인코딩한 뒤 다시 나눕니다.
```

In [ ]:
# 🔤 OHE: train+test를 합쳐서 인코딩 → 컬럼 어긋남 방지
combined = pd.concat([Xtrain, Xtest], axis=0, keys=['train', 'test'])
combined = pd.get_dummies(combined, columns=ohe_cols)

# 🏷 라벨인코딩: 종류가 많은 범주형은 숫자 하나로 치환
for c in label_cols:
    le = LabelEncoder()
    combined[c] = le.fit_transform(combined[c].astype(str))

Xtrain_enc = combined.loc['train'].copy()
Xtest_enc  = combined.loc['test'].copy()

print('인코딩 후 컬럼 수 - Xtrain:', Xtrain_enc.shape[1], ' Xtest:', Xtest_enc.shape[1])
Xtrain_enc.head()


### (7) 정답(Target) 인코딩 — 글자면 숫자로

```
정답이 'Good'/'Standard'/'Poor'처럼 글자면 라벨인코딩이 필요합니다.
이미 0/1 숫자면 그대로 사용합니다.
```

In [ ]:
# pandas 버전에 따라 문자열 컬럼이 'object' 대신 'str'로 표시될 수 있어
# dtype 문자열 비교 대신 is_numeric_dtype()으로 판단합니다.
if not pd.api.types.is_numeric_dtype(ytrain):
    le_target = LabelEncoder()
    ytrain_enc = le_target.fit_transform(ytrain)
    print('클래스 매핑:', dict(zip(le_target.classes_, range(len(le_target.classes_)))))
else:
    le_target = None
    ytrain_enc = ytrain.values
    print('정답이 이미 숫자라 인코딩 생략')


### (8) 스케일링 — 숫자 단위 통일

```
⚠️ 규칙: fit_transform(train) → train 기준 계산+변환 / transform(test) → 변환만!
```

In [ ]:
scaler = MinMaxScaler()

Xtrain_scaled = Xtrain_enc.copy()
Xtest_scaled  = Xtest_enc.copy()

if num_cols:
    Xtrain_scaled[num_cols] = scaler.fit_transform(Xtrain_enc[num_cols])
    Xtest_scaled[num_cols]  = scaler.transform(Xtest_enc[num_cols])

print('스케일링 완료!')
Xtrain_scaled.head()


---
## [Step4] AI 모델 학습 + 평가

### (1) train / val 분리 — stratify로 클래스 비율 유지

```
다중클래스일 때 무작위로 나누면 어떤 클래스가 val에 너무 적게 들어갈 수 있습니다.
stratify=ytrain_enc를 쓰면 train/val의 클래스 비율을 원본과 똑같이 맞춰줍니다.
```

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    Xtrain_scaled, ytrain_enc,
    test_size=0.2,
    stratify=ytrain_enc,   # ★ 클래스 비율 유지
    random_state=42
)
print('학습용:', X_train.shape, ' 검증용:', X_val.shape)


### (2) 여러 모델 학습 → 가장 좋은 모델 찾기 (f1-macro 기준)

```
★ 이진분류든 다중분류든 f1_score(average='macro')로 통일해서 비교합니다.
   accuracy도 참고용으로 같이 출력합니다.
```

In [ ]:
models = {
    'LogisticRegression':     LogisticRegression(max_iter=1000),
    'DecisionTreeClassifier': DecisionTreeClassifier(random_state=42),
    'KNN':                    KNeighborsClassifier(),
    'RandomForestClassifier': RandomForestClassifier(random_state=42, n_jobs=-1),
    'ExtraTreesClassifier':   ExtraTreesClassifier(random_state=42, n_jobs=-1),
    'XGBClassifier':          XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss' if n_classes > 2 else 'logloss'),
    'LGBMClassifier':         LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
}

selected_estimator = None
selected_name = None
max_score = -1

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_val)
    f1 = f1_score(y_val, pred, average='macro')
    acc = accuracy_score(y_val, pred)
    print(f'[{name:24}] f1-macro: {f1:.4f}  |  accuracy: {acc:.4f}')

    if f1 > max_score:
        selected_estimator, selected_name, max_score = estimator, name, f1
        print('  ★ 현재 최선!')

print('=' * 60)
print('최종 선택 모델:', selected_name, ' (f1-macro:', round(max_score, 4), ')')

PARAM_GUIDE = {
    'LGBMClassifier':         {'learning_rate': [0.05, 0.1, 0.14], 'n_estimators': [100, 200, 300]},
    'XGBClassifier':          {'learning_rate': [0.05, 0.1, 0.2],  'n_estimators': [100, 200, 300], 'max_depth': [3, 5, 7]},
    'RandomForestClassifier': {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'ExtraTreesClassifier':   {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10]},
    'DecisionTreeClassifier': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'KNN':                    {'n_neighbors': [3, 5, 7, 9]},
}
if selected_name in PARAM_GUIDE:
    print(f'\n[{selected_name}] 아래 파라미터를 GridSearchCV에 복붙하세요:')
    print('parameters =', PARAM_GUIDE[selected_name])


### (3) 하이퍼파라미터 튜닝 (GridSearchCV)

```
scoring='f1_macro' → ★ 이 템플릿의 평가지표와 통일
```

In [ ]:
# ★ 위에서 출력된 파라미터를 여기에 복붙!
parameters = PARAM_GUIDE.get(selected_name, {})

grid_clf = GridSearchCV(
    selected_estimator,
    param_grid=parameters,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
)
grid_clf.fit(Xtrain_scaled, ytrain_enc)

best_model = grid_clf.best_estimator_
print('최적 파라미터:', grid_clf.best_params_)
print('최적 CV 성능 (f1-macro):', round(grid_clf.best_score_, 4))


---
## [Step5] 테스트 데이터 예측 + 혼동행렬로 확인

```
혼동행렬(Confusion Matrix)은 다중클래스에서 특히 중요합니다.
어떤 클래스끼리 자주 헷갈리는지 한눈에 보여줍니다.
```

In [ ]:
val_pred = best_model.predict(X_val)
print(classification_report(y_val, val_pred))

labels = le_target.classes_ if le_target is not None else sorted(set(ytrain_enc))
cm = confusion_matrix(y_val, val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Validation Confusion Matrix')
plt.show()


In [ ]:
pred_test = best_model.predict(Xtest_scaled)

if le_target is not None:
    pred_test_label = le_target.inverse_transform(pred_test)
else:
    pred_test_label = pred_test

print('예측값 샘플:', pred_test_label[:10])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(x=ytrain, ax=axes[0]).set_title('Train 실제값 분포')
sns.countplot(x=pred_test_label, ax=axes[1]).set_title('Test 예측값 분포')
plt.tight_layout()
plt.show()


---
## [Step6] 제출 파일 만들기

In [ ]:
submission = pd.read_csv('./data/sample-submission.csv')
submission[TARGET] = pred_test_label
submission.to_csv('./data/MySubmission.csv', index=False)

print('✅ 제출 파일 저장 완료! → data/MySubmission.csv')
submission.head()


---
## [Step7] 성능 더 올리기 (선택)

```
아래는 시간이 남을 때 시도해볼 수 있는 개선 방법들입니다.
전부 다 할 필요는 없고, 필요한 것만 골라 쓰세요.
```

### (1) StratifiedKFold 교차검증 — 더 믿을 만한 성능 측정
```
train/val 한 번만 나누면 운 좋게(혹은 나쁘게) 나뉜 결과일 수 있습니다.
K번 나누어 평균을 내면 더 안정적인 성능 추정치를 얻습니다.
```

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, Xtrain_scaled, ytrain_enc, cv=skf, scoring='f1_macro', n_jobs=-1)
print('5-Fold f1-macro 점수들:', np.round(cv_scores, 4))
print('평균:', round(cv_scores.mean(), 4), ' 표준편차:', round(cv_scores.std(), 4))


### (2) 클래스 불균형 대응

```
한 클래스가 압도적으로 많으면(예: 정상 95% / 이상 5%) 모델이 다수 클래스만 찍어도
점수가 높게 나올 수 있습니다. 아래 옵션들로 소수 클래스에 가중치를 줍니다.
```

In [ ]:
# LogisticRegression / RandomForest 계열: class_weight='balanced'
rf_balanced = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

# LightGBM: is_unbalance=True (또는 class_weight='balanced')
lgbm_balanced = LGBMClassifier(is_unbalance=True, random_state=42, verbose=-1)

# XGBoost 이진분류: scale_pos_weight = (음성 개수 / 양성 개수)
# scale_pos_weight = (ytrain_enc == 0).sum() / (ytrain_enc == 1).sum()  # 이진분류일 때만
print('불균형이 심하면 위 옵션들을 모델 생성 시 넣어서 다시 학습해보세요.')


### (3) Early Stopping — 과적합 방지 + 튜닝 시간 단축 (XGBoost/LightGBM)

In [ ]:
# XGBoost / LightGBM은 검증셋 성능이 더 이상 좋아지지 않으면 학습을 일찍 멈출 수 있습니다.
early_model = LGBMClassifier(
    n_estimators=1000,     # 넉넉하게 크게 주고
    learning_rate=0.05,
    random_state=42,
    verbose=-1,
)
early_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[],  # 버전에 따라 lightgbm.early_stopping(30) 콜백을 넣어 조기 종료할 수 있습니다
)
print('Early stopping은 n_estimators를 크게 잡아도 자동으로 적당한 지점에서 멈추게 해줍니다.')
print('(LightGBM 버전에 따라 import lightgbm; callbacks=[lightgbm.early_stopping(30)] 형태로 사용)')


### (4) 파생변수(피처 엔지니어링) 아이디어

In [ ]:
# 예시: 두 숫자 컬럼의 비율/차이로 새로운 의미 있는 변수 만들기
# Xtrain_scaled['새컬럼'] = Xtrain['A컬럼'] / (Xtrain['B컬럼'] + 1e-6)
# Xtest_scaled['새컬럼']  = Xtest['A컬럼']  / (Xtest['B컬럼']  + 1e-6)

# 예시: 연속형 변수를 구간화(binning)해서 범주형처럼 사용
# Xtrain_scaled['Age_bin'] = pd.cut(Xtrain['Age'], bins=[0,20,40,60,100], labels=False)

print('데이터 도메인 지식이 있다면 위 방식으로 파생변수를 만들어 재학습해보세요.')


### (5) 탐색 범위 넓히기 — RandomizedSearchCV (조합이 너무 많을 때)

In [ ]:
wide_parameters = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 7, 10, 15],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=wide_parameters,
    n_iter=10,          # 전체 조합 중 10개만 무작위로 시도 → GridSearch보다 빠름
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=-1,
)
random_search.fit(Xtrain_scaled, ytrain_enc)
print('RandomizedSearch 최적 파라미터:', random_search.best_params_)
print('RandomizedSearch 최적 성능:', round(random_search.best_score_, 4))


### (6) 피처 중요도 기반 가지치기

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    order = np.argsort(importances)[::-1]

    plt.figure(figsize=(8, 4))
    top_n = min(15, len(order))
    sns.barplot(x=importances[order[:top_n]], y=Xtrain_scaled.columns[order[:top_n]])
    plt.title('Top Feature Importances')
    plt.tight_layout()
    plt.show()

    # 중요도가 낮은 피처를 제거하고 다시 학습 → 성능이 비슷하거나 좋아지면 채택
    low_importance_cols = Xtrain_scaled.columns[order[top_n:]]
    print(f'중요도 하위 {len(low_importance_cols)}개 컬럼은 제거 후 재학습을 시도해볼 수 있습니다.')
else:
    print('이 모델은 feature_importances_를 지원하지 않습니다 (예: LogisticRegression, KNN).')


### (7) 간단 앙상블 — 상위 모델들을 soft-voting으로 결합

In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(random_state=42, n_jobs=-1)),
        ('xgb', XGBClassifier(random_state=42, n_jobs=-1, eval_metric='mlogloss' if n_classes > 2 else 'logloss')),
        ('lgbm', LGBMClassifier(random_state=42, verbose=-1)),
    ],
    voting='soft',   # 각 모델의 확률을 평균내서 최종 클래스 결정
)
voting_model.fit(X_train, y_train)
voting_pred = voting_model.predict(X_val)
print('Voting Ensemble f1-macro:', round(f1_score(y_val, voting_pred, average='macro'), 4))
print('단일 최선 모델 f1-macro  :', round(max_score, 4))
print('→ 앙상블 점수가 더 높으면 최종 제출을 voting_model 예측으로 바꿔도 좋습니다.')


---
## 🚨 자주 하는 실수 체크리스트

```
1. test에 fit_transform() 사용                        ❌ → transform()만 사용
2. Target 컬럼을 feature(X)에 포함                     ❌ → drop([TARGET]) 먼저!
3. train/test를 따로 get_dummies → 컬럼 수가 달라짐    ❌ → 합쳐서 인코딩 후 분리
4. 다중분류인데 accuracy만 확인                        ❌ → f1_score(average='macro') 확인
5. train 따로 test 따로 결측치 기준값 계산              ❌ → 항상 train 기준값을 test에도 적용
```
